# Laboratory 8: QTris — Quantum Mechanics as a Board Game

QTris plays tic-tac-toe on a 3×3 grid whose nine cells are nine qubits: tiles are states,
move cards are unitaries, and the endgame is a dice-resolved measurement. The interactive
lab guide lets you *play* the game; this notebook rebuilds its state space as **quantum
circuits** and audits the game's printed measurement tables against the Born rule, seeded
end to end. The questions driving the six parts:

1. **The tiles** — a half-white-half-black tile turns up white half the time, like a coin.
   Why does the game print an *orientation* on it — what could the direction of a 50/50
   tile change?
2. **The pairs** — two triangle tiles always agree on color though each alone is a perfect
   coin toss. Where does the agreement live?
3. **The printed tables** — the rulebook ships forty dice rows. How close are printed
   integer bands to exact quantum probabilities?
4. **The erratum** — two of those rows turn out to be *impossible* for any gate. Can a
   notebook prove that, rather than assert it?
5. **The dice** — what changes between rolling the printed d100 bands and sampling the
   exact probabilities?
6. **Two ways to measure orientation** — same statistics, different post-measurement
   states. What exactly differs?

**About the original game.** *QTris e Meccanica Quantistica* was created by Alioscia Hamma
(concept and physics) with the creative team Immacolata De Simone, Michela Nazzaro and
Michele Viscardi (illustrations M. Viscardi), produced by AH Quantum Lab and Communikart,
and published online in 2024 by NQSTI — the National Quantum Science and Technology
Institute ([nqsti.it](https://www.nqsti.it)). This notebook is part of an educational
adaptation built with full attribution; every deviation from the printed game is declared
in the lab guide's deviations section. The rulebook's own claim — *there is no fundamental
difference between QTris and quantum mechanics* — is exactly what this notebook checks.

> ### Convention note — bit ordering (it genuinely matters in this lab)
>
> This lab series reports every quantum state in the **lab convention (big-endian)**: in
> $|q_0 q_1 \dots q_{n-1}\rangle$, qubit $0$ is the **leftmost** bit. **Qiskit uses the
> opposite (little-endian) order**, with qubit $0$ rightmost in returned bitstrings.
>
> Every entangled pair in QTris is a **two-qubit register**, with qubit $0$ = the
> **lower-numbered cell** (the first letter of outcomes like $WB$). So every displayed
> two-bit string in this notebook goes through `qiskit_to_lab()` — a pure string reversal —
> and every state vector through `statevector_lab()`. Where you see $WB$, it means
> *lower cell white, higher cell black*.

### Environment Setup

In [ ]:
"""
Environment Setup Module.
Run this cell first to install all required libraries.
"""
%pip install -q qiskit qiskit-aer

import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

# All randomness in this notebook is seeded, so every run reproduces these numbers.
SEED = 42


def qiskit_to_lab(bitstring: str) -> str:
    """Qiskit little-endian bitstring -> lab big-endian order (string reversal).

    The lab convention puts qubit 0 leftmost; for pair registers qubit 0 is
    the lower-numbered cell, so the first letter of a converted string is
    that cell's color.
    """
    return bitstring[::-1]


def lab_to_qiskit(bitstring: str) -> str:
    """Inverse conversion (also a string reversal)."""
    return bitstring[::-1]


def statevector_lab(qc: QuantumCircuit) -> np.ndarray:
    """Statevector amplitudes reordered to the lab big-endian convention."""
    return Statevector(qc).reverse_qargs().data


def sample_counts(qc: QuantumCircuit, shots: int, seed: int) -> dict:
    """Run a measured circuit on the seeded Aer simulator -> counts dict."""
    sim = AerSimulator(seed_simulator=seed)
    return sim.run(transpile(qc, sim, seed_transpiler=seed), shots=shots).result().get_counts()


def lab_counts(counts: dict) -> dict:
    """Counts with every bitstring converted to lab order, keys sorted."""
    return dict(sorted(((qiskit_to_lab(k), v) for k, v in counts.items())))


print("Setup complete. Base seed:", SEED)

---
## 1 · The Tile Dictionary and the Pinned Rotation

**The question first:** *a half-white-half-black tile turns up white exactly half the
time — so does a coin. Why does the game print an orientation on it?*

**Why this step:** everything in QTris rests on four tile states and one hidden gate. The
tiles are $|W\rangle = |0\rangle$, $|B\rangle = |1\rangle$, and the two *bianconere*
$|R\rangle = |+\rangle$ (white half on the **right**) and $|L\rangle = |-\rangle$. The
pink-square marker is a gate the rulebook never writes down — but its printed measurement
rows (white 25, black 75, right-bianconera 93, left-bianconera 7) pin it completely: it is
the **canonical rotation**

$$U = R(-60°) = \begin{pmatrix} 1/2 & \sqrt{3}/2 \\ -\sqrt{3}/2 & 1/2 \end{pmatrix},$$

because $\cos^2 60° = 1/4$ forces the angle and the 93 landing on the *right*-oriented
tile fixes the sense. As a circuit gate, $U = R_y(-2\pi/3)$. A tidy closed form:
$U|+\rangle = R(-15°)|0\rangle$, so $P(W \mid U \text{ on } R) = \cos^2 15° =
\tfrac{2+\sqrt{3}}{4}$ — the game's 93 carries a $\sqrt{3}$.

**Float64 rule.** Every expected value below is either exactly representable (like $1/2$,
$1/4$, $3/8$) or declared as the float64 evaluation of its defining expression with a
17-digit literal — e.g. $\tfrac{2+\sqrt{3}}{4} \to$ `0.9330127018922193`. Different
computation paths may land one ulp apart (the printed `repr` values let you see it);
agreement is always asserted at $10^{-12}$.

In [ ]:
"""
Part 1 - the eight tile states as circuits; the printed single-cell rows.

The pink square U = R(-60deg) is pinned entrywise BEFORE use, then applied
as the circuit gate ry(-2pi/3). Preparation circuits: B = x; R = h;
L = x, h  (H|1> = |->); a pink square appends ry(-2pi/3) last (the marker
sits outermost on the tile).
"""
U_MATRIX = np.array([[0.5, np.sqrt(3) / 2], [-np.sqrt(3) / 2, 0.5]])

qc_u = QuantumCircuit(1)
qc_u.ry(-2 * np.pi / 3, 0)
defect = float(np.max(np.abs(Operator(qc_u).data - U_MATRIX)))
assert defect < 1e-12
print(f"U pinned entrywise; ry(-2pi/3) equivalence defect = {defect:.3e}")

P_W_UR = float(np.cos(np.pi / 12) ** 2)      # cos^2 15deg = (2 + sqrt(3))/4
print("P(W | U on R) = cos^2 15deg =", repr(P_W_UR), "  declared literal: 0.9330127018922193")
assert abs(P_W_UR - 0.9330127018922193) < 1e-12


def single_cell_circuit(name: str) -> QuantumCircuit:
    """Preparation circuit of a tile state ('W', 'B', 'R', 'L', 'UW', ...)."""
    qc = QuantumCircuit(1)
    base = name[-1]
    if base == "B":
        qc.x(0)
    elif base == "R":
        qc.h(0)
    elif base == "L":
        qc.x(0)
        qc.h(0)
    if len(name) == 2:                       # the pink square, outermost
        qc.ry(-2 * np.pi / 3, 0)
    return qc


# The printed single-cell dice rows (white-band length out of 100), transcribed
# from the rulebook's table as data:
PRINTED_SINGLE = {"W": 100, "B": 0, "R": 50, "L": 50, "UW": 25, "UB": 75, "UR": 93, "UL": 7}

print()
print("state   P(W) exact              printed  |print - exact|")
max_err = 0.0
for name, band in PRINTED_SINGLE.items():
    p_white = float(np.abs(Statevector(single_cell_circuit(name)).data[0]) ** 2)
    err = abs(band / 100 - p_white)
    max_err = max(max_err, err)
    print(f"{name:5s}   {p_white!r:22s}  {band:3d}      {err:.6f}")
print(f"largest print-vs-physics gap over the 8 single rows: {max_err:.6f}")
assert max_err < 0.005                        # all single rows conform well inside 0.005

---
## 2 · Bell Pairs, Prepared the Way the Game Prepares Them

**The question first:** *two triangles on opposite corners always come up the same color,
though each alone is a perfect coin toss. Where does the agreement live?*

**Why this step:** the game's only two-cell card, $C_X$, is a controlled-NOT with the
control declared by the player. Played with a bianconera control and a monochrome target
it entangles the two cells into a Bell state by a simple recipe — **orientation ↦ sign,
target color ↦ same-color ($\Phi$) or opposite-color ($\Psi$) type**:

$$(R, W) \to \Phi^+ \qquad (R, B) \to \Psi^+ \qquad (L, W) \to \Phi^- \qquad
  (L, B) \to \Psi^-$$

The circuits below are literally that recipe: prepare the control tile on qubit $0$ (the
lower-numbered cell), the target tile on qubit $1$, then one `cx`. Remember the
conversion: every amplitude vector and bitstring is displayed in **lab order**.

In [ ]:
"""
Part 2 - the four Bell cores by the game's own recipe; their measurement rows.

Amplitude convention (lab order, index = 2*q0 + q1, q0 = lower cell):
Phi+ = [1, 0, 0, 1]/sqrt2      Psi+ = [0, 1, 1, 0]/sqrt2
Phi- = [1, 0, 0, -1]/sqrt2     Psi- = [0, 1, -1, 0]/sqrt2
The 1/sqrt2 literal is 0.7071067811865476 (float64 of the defining expression).
"""
R2 = 0.7071067811865476
BELL_EXPECTED = {
    "PhiPlus":  np.array([R2, 0.0, 0.0, R2]),
    "PsiPlus":  np.array([0.0, R2, R2, 0.0]),
    "PhiMinus": np.array([R2, 0.0, 0.0, -R2]),
    "PsiMinus": np.array([0.0, R2, -R2, 0.0]),
}
PAIR_PREPS = {"PhiPlus": ("R", "W"), "PsiPlus": ("R", "B"),
              "PhiMinus": ("L", "W"), "PsiMinus": ("L", "B")}


def pair_circuit(core: str) -> QuantumCircuit:
    """The C_X card's preparation: control tile on q0, target on q1, then cx."""
    control, target = PAIR_PREPS[core]
    qc = QuantumCircuit(2)
    if control == "L":
        qc.x(0)
    qc.h(0)
    if target == "B":
        qc.x(1)
    qc.cx(0, 1)
    return qc


OUTCOMES = ["WW", "WB", "BW", "BB"]
for core, expected in BELL_EXPECTED.items():
    amps = np.real(statevector_lab(pair_circuit(core)))
    defect = float(np.max(np.abs(amps - expected)))
    probs = np.abs(statevector_lab(pair_circuit(core))) ** 2
    control, target = PAIR_PREPS[core]
    print(f"({control},{target}) -> {core:9s} amplitude defect vs literals: {defect:.3e}   "
          f"P(WW,WB,BW,BB) = [{', '.join(f'{p:.4f}' for p in probs)}]")
    assert defect < 1e-12
    # structural zeros are EXACT zeros of the amplitudes, not small numbers:
    if core.startswith("Phi"):
        assert probs[1] == 0.0 and probs[2] == 0.0
    else:
        assert probs[0] == 0.0 and probs[3] == 0.0
print("\nPerfect (anti)correlation with individually random members - the agreement")
print("lives in the pair, not in either cell. Mixed outcomes have probability exactly 0.")

---
## 3 · Auditing the Printed Pair Tables

**The question first:** *the rulebook prints 32 dice rows for decorated pairs. How close
is print to physics?*

**Why this step:** markers under a pair member are gates on that member's qubit: the red
H square is the $H$ gate applied **first** (it sits against the tile), the pink U square
is $R_y(-2\pi/3)$ applied **last** (it sits outermost) — this ordering is not a choice;
it is the unique reading that reproduces the printed 47/3 rows, since
$|\langle 0|UH|0\rangle|^2 = \tfrac{2+\sqrt{3}}{8} \ne
|\langle 0|HU|0\rangle|^2$. The audit below computes every configuration's exact
distribution from the state vector and compares it with the printed integer bands,
transcribed as data. Two rows are flagged: they are the subject of Part 4.

*Counting note:* the game's full measurement table has **40 printed rows** — the 8
single-cell rows audited in Part 1 plus the 32 pair rows audited here. Together the two
audits give the count quoted throughout the lab guide: **38 of 40 rows conform** within
the print's 0.005 rounding (all 8 single rows conform, with a largest gap of 0.0030; 30
of the 32 pair rows conform, attaining 0.005), and the 2 flagged pair rows are the
erratum of Part 4.

In [ ]:
"""
Part 3 - the 32 printed pair rows vs exact Born probabilities.

PRINTED_PAIR maps (stack-on-member-1, stack-on-member-2, core) to the printed
band lengths over (WW, WB, BW, BB) out of 100 - a literal transcription of the
rulebook's tables (the audit is print-vs-physics, so the print is data here,
never derived). The two rows marked e2 are printed as if two pink squares
always cancel; Part 4 proves no gate can do that.
"""
CORES = ["PhiPlus", "PsiPlus", "PhiMinus", "PsiMinus"]
CONFIGS = [("I", "I"), ("H", "I"), ("U", "I"), ("UH", "I"),
           ("I", "U"), ("H", "U"), ("U", "U"), ("UH", "U")]
corr, anti = (50, 0, 0, 50), (0, 50, 50, 0)
unif, e8, e8m = (25, 25, 25, 25), (12, 38, 38, 12), (38, 12, 12, 38)
g, gm = (47, 3, 3, 47), (3, 47, 47, 3)
PRINTED_PAIR = {
    ("I", "I"):  [corr, anti, corr, anti],
    ("H", "I"):  [unif, unif, unif, unif],
    ("U", "I"):  [e8, e8m, e8, e8m],
    ("UH", "I"): [g, gm, g, gm],
    ("I", "U"):  [e8, e8m, e8, e8m],
    ("H", "U"):  [g, g, gm, gm],
    ("U", "U"):  [corr, anti, corr, anti],     # rows 2-3 are the erratum (Part 4)
    ("UH", "U"): [gm, unif, unif, g],
}
E2_ROWS = {("U", "U", "PsiPlus"), ("U", "U", "PhiMinus")}


def decorated_pair_circuit(core: str, s1: str, s2: str) -> QuantumCircuit:
    qc = pair_circuit(core)
    for qubit, stack in ((0, s1), (1, s2)):
        if "H" in stack:
            qc.h(qubit)                       # red square: core-adjacent, first
        if "U" in stack:
            qc.ry(-2 * np.pi / 3, qubit)      # pink square: outermost, last
    return qc


max_conforming, max_flagged = 0.0, 0.0
n_conforming = 0
print("stacks    core       exact (WW WB BW BB)              printed     max|diff|")
for s1, s2 in CONFIGS:
    for core, bands in zip(CORES, PRINTED_PAIR[(s1, s2)]):
        probs = np.abs(statevector_lab(decorated_pair_circuit(core, s1, s2))) ** 2
        err = float(np.max(np.abs(np.array(bands) / 100 - probs)))
        flagged = (s1, s2, core) in E2_ROWS
        if flagged:
            max_flagged = max(max_flagged, err)
        else:
            max_conforming = max(max_conforming, err)
            n_conforming += 1
        exact_str = " ".join(f"{p:.4f}" for p in probs)
        band_str = "/".join(str(b) for b in bands)
        print(f"({s1:2s},{s2:1s})   {core:9s}  [{exact_str}]   {band_str:11s} "
              f"{err:.4f}{'  <- FLAGGED (Part 4)' if flagged else ''}")

print()
print(f"conforming rows: {n_conforming} of 32; max |print - exact| =", repr(max_conforming))
print("the bound 0.005 is ATTAINED - the authors rounded 12.5/37.5 in opposite")
print("directions (12 and 38), keeping every row total and both marginals exact.")
assert abs(max_conforming - 0.005) < 1e-12 and max_conforming > 0.004
print(f"flagged rows: 2 of 32; max |print - exact| =", repr(max_flagged), "(= 3/8, Part 4)")
assert abs(max_flagged - 0.375) < 1e-12

---
## 4 · The Erratum, Computed — Not Asserted

**The question first:** *can two rows of a published table be provably impossible — for
every possible gate at once?*

**Why this step:** the flagged rows print the double-U configurations of $\Psi^+$ and
$\Phi^-$ as unchanged (perfect anti/correlation), as if two pink squares always cancel.
For $\Phi^+$ and the singlet $\Psi^-$ that *is* exact physics — those two states are
invariant under a common rotation of both members. But suppose the $\Phi^+$ **and**
$\Phi^-$ rows were both right as printed. Then the square $U$ would have to satisfy both
invariances at once, which forces $U Z U^{T}$ to stay diagonal — and the size of its
off-diagonal element is a single computable number:

$$\bigl|(U Z U^{T})_{01}\bigr|^2 = \sin^2 120° = \tfrac34 \ne 0.$$

The printed rows need this to be $0$; it is $3/4$ — no unitary assignment to the pink
square survives (the same argument rules out any *pair* of gates). Rotational invariance
is a property of **specific states**, not of entanglement in general — and the game's own
tables, exactly right in 38 of 40 rows, over-generalized it in these two. A remarkable
catch: formalism referees everyone, including the authors of an excellent game.

In [ ]:
"""
Part 4 - the obstruction number, the exact erratum rows, and the two genuine
invariances, all computed from the pinned U.
"""
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
obstruction = float(np.abs((U_MATRIX @ Z @ U_MATRIX.T)[0, 1]) ** 2)
print("|(U Z U^T)_01|^2 =", repr(obstruction), " (exact value 3/4; the printed rows need 0)")
assert abs(obstruction - 0.75) < 1e-12

# The two genuine invariances (exact physics, printed correctly):
UU = np.kron(U_MATRIX, U_MATRIX)
for core in ("PhiPlus", "PsiMinus"):
    psi = np.real(statevector_lab(pair_circuit(core)))
    overlap = float(abs(np.dot(psi, UU @ psi)))
    print(f"|<{core}| U(x)U |{core}>| = {overlap!r}  (invariant up to a global sign)")
    assert abs(overlap - 1.0) < 1e-12

# The two erratum rows, exact vs printed:
print()
print("exact double-U distributions (both dice modes in the game serve these):")
for core, printed in (("PsiPlus", (0, 50, 50, 0)), ("PhiMinus", (50, 0, 0, 50))):
    probs = np.abs(statevector_lab(decorated_pair_circuit(core, "U", "U"))) ** 2
    dev = float(np.max(np.abs(np.array(printed) / 100 - probs)))
    print(f"  {core:9s}: [{', '.join(repr(round(float(p), 15)) for p in probs)}]"
          f"   printed {'/'.join(str(b) for b in printed)}   deviation {dev!r}")
    assert abs(dev - 0.375) < 1e-12
print("\ndeviation exactly 3/8 in every outcome - not a rounding of anything.")

---
## 5 · Dice as Sampling: Seeded Runs vs Exact Probabilities

**The question first:** *what changes between rolling the printed d100 bands and sampling
the exact probabilities?*

**Why this step:** the game's measurement phase is a hand-executed Born sampler — the same
physics that commercial quantum random-number generators package in a chip. The lab guide's
*printed-tables* mode reproduces the published integer bands (within their documented
$\le 0.005$ rounding); its *exact* mode — and this notebook — samples the full-precision
probabilities. Below, seeded simulator runs sit beside the exact numbers. Note the
structural zeros: the $\Phi^+$ pair's mixed outcomes are not merely rare, they are
**absent** — a $10^4$-shot run contains exactly zero of them, because the amplitude is
exactly zero (and the printed row, correctly, has no band for them at all).

In [ ]:
"""
Part 5 - seeded Aer sampling vs exact values (distinct seed per experiment).
"""
shots = 10_000

# (a) the decorated right-bianconera: the game's famous 93
qc = single_cell_circuit("UR")
qc.measure_all()
frac_white = sample_counts(qc, shots, SEED + 1).get("0", 0) / shots
print(f"U on R: sampled white fraction {frac_white:.4f}   exact {P_W_UR!r}")
assert abs(frac_white - P_W_UR) < 0.01        # 4 sigma for p ~ 0.933, n = 10^4

# (b) structural zeros on the plain Phi+ pair
qc = pair_circuit("PhiPlus")
qc.measure_all()
counts = lab_counts(sample_counts(qc, shots, SEED + 2))
print("Phi+ pair, lab-order counts:", counts)
assert counts.get("01", 0) == 0 and counts.get("10", 0) == 0   # WB and BW
print("mixed outcomes over 10^4 shots: 0 and 0 - structurally absent, not rare")

# (c) the corrected double-U Phi- row, sampled
qc = decorated_pair_circuit("PhiMinus", "U", "U")
qc.measure_all()
counts = lab_counts(sample_counts(qc, shots, SEED + 3))
fracs = [counts.get(b, 0) / shots for b in ("00", "01", "10", "11")]
print(f"double-U Phi-: sampled [{', '.join(f'{f:.4f}' for f in fracs)}]   "
      f"exact [0.125, 0.375, 0.375, 0.125]")
for f, p in zip(fracs, [0.125, 0.375, 0.375, 0.125]):
    assert abs(f - p) < 0.02                  # ~4 sigma per outcome
print("the dice agree with the corrected row, not with the printed one.")

---
## 6 · Two Ways to Measure Orientation

**The question first:** *you measured the tile white; measure orientation, then color
again — half your whites turned black. Who repainted the tile?*

**Why this step:** the *orientation* measurement asks the $|+\rangle/|-\rangle$
question, and the lab names its two implementations as in Lab 5. **Procedure 1** projects
directly onto the orientation states; the post-measurement state is an oriented bianconera.
**Procedure 2** is the rulebook's own apparatus: apply $H$ — the canonical rotation for
this basis — measure *color*, and relabel the outcomes; the post-measurement state is a
monochrome tile. Identical statistics (that is a theorem, checked below on a state where
the outcome probabilities are *not* $1/2$), different post-states — their overlap is
$|\langle + | 0 \rangle|^2 = 1/2$. Color and orientation are mutually unbiased — the
same incompatible-question pair that ran Lab 6's key distribution, and the reason the
chain W → orientation → color → orientation is 50/50 at every step: Stern-Gerlach physics,
the readout principle of today's spin qubits.

In [ ]:
"""
Part 6 - Procedures 1 and 2 on U|+> (asymmetric on purpose), and the
incompatibility chain from |W>.
"""
plus = np.array([1.0, 1.0]) / np.sqrt(2)
minus = np.array([1.0, -1.0]) / np.sqrt(2)
ket_w = np.array([1.0, 0.0])

psi_ur = Statevector(single_cell_circuit("UR")).data

# Procedure 1: direct projection onto the orientation states
p1 = [float(np.abs(np.vdot(plus, psi_ur)) ** 2), float(np.abs(np.vdot(minus, psi_ur)) ** 2)]

# Procedure 2: apply H (the canonical rotation), read color, relabel
qc = single_cell_circuit("UR")
qc.h(0)
probs_h = np.abs(Statevector(qc).data) ** 2
p2 = [float(probs_h[0]), float(probs_h[1])]

print(f"orientation of U|+>: Procedure 1 [{p1[0]!r}, {p1[1]!r}]")
print(f"                     Procedure 2 [{p2[0]!r}, {p2[1]!r}]")
assert max(abs(a - b) for a, b in zip(p1, p2)) < 1e-12
print("identical statistics (and NOT 50/50 - the agreement is non-trivial here)")

cross = float(np.abs(np.vdot(plus, ket_w)) ** 2)
print("post-state overlap |<+|0>|^2 =", repr(cross), " (bianconera tile vs monochrome tile)")
assert abs(cross - 0.5) < 1e-12

# The chain from |W>: orientation -> color -> orientation, conditioning on the
# first-listed outcome each step (post-states: oriented tile, then plain tile).
steps = []
p = [float(np.abs(np.vdot(plus, ket_w)) ** 2), float(np.abs(np.vdot(minus, ket_w)) ** 2)]
steps.append(("orientation on W", p))
p = [float(np.abs(plus[0]) ** 2), float(np.abs(plus[1]) ** 2)]
steps.append(("color on the |+> tile", p))
p = [float(np.abs(np.vdot(plus, ket_w)) ** 2), float(np.abs(np.vdot(minus, ket_w)) ** 2)]
steps.append(("orientation on the W tile", p))
print()
for label, p in steps:
    print(f"{label:28s} [{p[0]!r}, {p[1]!r}]")
    assert max(abs(x - 0.5) for x in p) < 1e-12
print("50/50 at every step: measuring one property fully randomizes the other.")

---
## Wrap-up — Back to the Board

Every number in this notebook is a number you can *roll* in the lab guide's game:

- the 50/50 bianconera tally and the $H$-card certainty (Parts 1–2 → play preset
  +C$_X$ and try the rulebook's own $H$-then-measure tactic);
- the pair counter pinned at 100% agreement with mixed outcomes structurally absent
  (Part 2 → entangle two far-apart cells and watch the counters);
- the printed 93 vs the exact $\cos^2 15°$ (Parts 1, 5 → flip the dice toggle between
  printed-tables mode and exact mode);
- the double-U erratum served as exact eighths in **both** dice modes (Part 4 → the
  guide's erratum panel, and play-task P2);
- the two orientation procedures with identical statistics and different post-states
  (Part 6 → the sandbox tab).

The adaptation's five declared deviations from the printed game — ambiguity resolutions,
the exact mode, the English translation, the reserved seam for a computer opponent, and
the erratum rows not being playable as printed — are listed in full in the lab guide.

**Attribution.** *QTris e Meccanica Quantistica* © 2024 AH Quantum Lab / Communikart —
Alioscia Hamma; Immacolata De Simone, Michela Nazzaro, Michele Viscardi; published online
by NQSTI ([nqsti.it](https://www.nqsti.it)). Further reading: M. Nielsen & I. Chuang,
*Quantum Computation and Quantum Information* (Cambridge UP, 2010) for the Bell/CNOT
formalism; Z. C. Seskir *et al.*, "Quantum games and interactive tools for quantum
technologies outreach and education," *Optical Engineering* **61**(8), 081809 (2022) for
where games like QTris sit in quantum education.